# Insights Quality Analysis

Evaluates the quality and behaviour of the clinical insight extraction pipeline.

**What this notebook covers:**
1. **Guardrail effectiveness** — how many claims were dropped by the evidence guardrail (hallucination-rate proxy)
2. **Schema adherence** — did the model return the expected structured fields?
3. **Claim category distribution** — which fields (risk_flags, diagnostic_hypotheses, recommended_followup) are populated
4. **Cross-run comparison** — do the two STT models (baseline vs candidate) produce materially different insights from the same audio?

Set `CANDIDATE_RUN_ID` and `BASELINE_RUN_ID` via environment variables or edit the cell below.

In [ ]:
import glob
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

CANDIDATE_RUN_ID = os.environ.get("CANDIDATE_RUN_ID", "9835ed61-9906-449d-bb8b-365cf7b0e70c")
BASELINE_RUN_ID  = os.environ.get("BASELINE_RUN_ID",  "87bac619-f810-4994-8a27-7b883b8f99b7")
INSIGHTS_DIR = Path("data/processed/insights_extract")

print(f"Candidate : {CANDIDATE_RUN_ID}")
print(f"Baseline  : {BASELINE_RUN_ID}")
print(f"Insights dir: {INSIGHTS_DIR.resolve()}")

In [ ]:
# ---------------------------------------------------------------------------
# Load insights artifacts from disk
# Resolves the latest artifact for each run_id.
# ---------------------------------------------------------------------------

def load_latest_artifact(run_id: str, insights_dir: Path) -> dict | None:
    pattern = str(insights_dir / f"insights_extract_{run_id}_*.json")
    matches = sorted(glob.glob(pattern))
    if not matches:
        return None
    with open(matches[-1], encoding="utf-8") as f:
        return json.load(f)


candidate_artifact = load_latest_artifact(CANDIDATE_RUN_ID, INSIGHTS_DIR)
baseline_artifact  = load_latest_artifact(BASELINE_RUN_ID,  INSIGHTS_DIR)

if candidate_artifact is None:
    print(f"No artifact found for candidate run {CANDIDATE_RUN_ID}")
if baseline_artifact is None:
    print(f"No artifact found for baseline run {BASELINE_RUN_ID}")

artifacts = {}
if candidate_artifact:
    artifacts["candidate"] = candidate_artifact
if baseline_artifact:
    artifacts["baseline"] = baseline_artifact

print(f"\nLoaded {len(artifacts)} artifact(s)")
for label, art in artifacts.items():
    print(f"  {label}: {art.get('samples_processed', 0)} samples, model={art.get('insight_model', '?')}")

In [ ]:
# ---------------------------------------------------------------------------
# Flatten results into a DataFrame: one row per (run_label, sample_id)
# ---------------------------------------------------------------------------

STRUCTURED_FIELDS = ["risk_flags", "diagnostic_hypotheses", "recommended_followup", "symptoms"]

rows = []
for label, art in artifacts.items():
    for result in art.get("results", []):
        insights = result.get("insights", {})
        guardrail = insights.get("guardrail", {})
        timing = result.get("timing", {})

        # Schema adherence: check whether the model returned the expected structured fields
        has_risk_flags_field   = "risk_flags" in insights
        has_diag_hyp_field     = "diagnostic_hypotheses" in insights
        has_followup_field     = "recommended_followup" in insights
        has_symptoms_field     = "symptoms" in insights
        # Non-standard keys the model may have used instead
        extra_keys = [k for k in insights if k not in (
            "clinical_presentation", "risk_flags", "diagnostic_hypotheses",
            "recommended_followup", "symptoms", "confidence",
            "evidence", "guardrail", "parse_error"
        )]

        row = {
            "run_label": label,
            "run_id": result.get("run_id"),
            "sample_id": result.get("sample_id"),
            "insight_model": result.get("insight_model"),
            "prompt_version": result.get("prompt_version"),
            # Guardrail
            "dropped_claims": guardrail.get("dropped_unsupported_claims", 0),
            # Claim counts (after guardrail)
            "n_risk_flags": len(insights.get("risk_flags", []) or []),
            "n_diagnostic_hypotheses": len(insights.get("diagnostic_hypotheses", []) or []),
            "n_recommended_followup": len(insights.get("recommended_followup", []) or []),
            "n_symptoms": len(insights.get("symptoms", []) or []),
            "has_clinical_presentation": bool(insights.get("clinical_presentation", "").strip()),
            "confidence": insights.get("confidence"),
            "has_parse_error": "parse_error" in insights,
            # Schema adherence
            "schema_compliant": all([has_risk_flags_field, has_diag_hyp_field, has_followup_field, has_symptoms_field]),
            "non_standard_keys": extra_keys,
            "n_non_standard_keys": len(extra_keys),
            # Timing
            "elapsed_s": timing.get("elapsed_s"),
            "word_count": timing.get("transcript_word_count"),
        }
        rows.append(row)

df = pd.DataFrame(rows)
print(f"Rows: {len(df)}")
display(df[["run_label", "sample_id", "dropped_claims", "n_risk_flags",
            "n_diagnostic_hypotheses", "n_recommended_followup", "n_symptoms",
            "schema_compliant", "has_parse_error"]])

In [ ]:
# ---------------------------------------------------------------------------
# Guardrail effectiveness summary
# ---------------------------------------------------------------------------

print("Guardrail Drop-Rate Summary")
print("-" * 40)
if df.empty:
    print("No data.")
else:
    total_claims_dropped = df["dropped_claims"].sum()
    total_claims_kept = (
        df["n_risk_flags"].sum() +
        df["n_diagnostic_hypotheses"].sum() +
        df["n_recommended_followup"].sum()
    )
    total_claims_attempted = total_claims_kept + total_claims_dropped
    drop_rate = (total_claims_dropped / total_claims_attempted) if total_claims_attempted > 0 else 0.0

    print(f"Total claims attempted (kept + dropped) : {total_claims_attempted}")
    print(f"Claims kept (evidence-grounded)         : {total_claims_kept}")
    print(f"Claims dropped (no quote support)       : {total_claims_dropped}")
    print(f"Guardrail drop rate                     : {drop_rate:.1%}")
    print()
    print("Per-run breakdown:")
    display(
        df.groupby("run_label")[["dropped_claims", "n_risk_flags",
                                 "n_diagnostic_hypotheses", "n_recommended_followup"]]
          .sum()
    )

    print()
    n_schema_issues = (~df["schema_compliant"]).sum()
    print(f"Schema-compliant outputs : {df['schema_compliant'].sum()} / {len(df)}")
    if n_schema_issues > 0:
        print(f"\n⚠  {n_schema_issues} sample(s) used non-standard keys instead of the expected schema.")
        print("Non-standard keys found:")
        for _, r in df[~df["schema_compliant"]].iterrows():
            print(f"  sample={r['sample_id']} extra_keys={r['non_standard_keys']}")
        print()
        print("Root cause: the model did not follow the JSON schema in the prompt.")
        print("Mitigation options:")
        print("  1. Add Ollama 'format' parameter with a JSON schema to enforce output structure.")
        print("  2. Use a larger model (8B+) which has better instruction-following.")
        print("  3. Add a validation + retry loop: if schema check fails, re-prompt with explicit error feedback.")

In [ ]:
# ---------------------------------------------------------------------------
# Claim category distribution plot
# ---------------------------------------------------------------------------
if not df.empty:
    category_cols = ["n_risk_flags", "n_diagnostic_hypotheses", "n_recommended_followup", "n_symptoms"]
    cat_labels = ["risk_flags", "diag_hypotheses", "followup", "symptoms"]

    for run_label, group in df.groupby("run_label"):
        totals = [group[c].sum() for c in category_cols]
        fig, ax = plt.subplots(figsize=(7, 4))
        bars = ax.bar(cat_labels, totals, color=["tomato", "steelblue", "seagreen", "orchid"])
        ax.bar_label(bars, padding=3)
        ax.set_title(f"Claim Category Distribution — {run_label} run")
        ax.set_ylabel("Total claims (post-guardrail)")
        ax.set_ylim(0, max(totals or [1]) + 2)
        plt.tight_layout()
        plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Cross-run insight consistency
# How similar are the clinical_presentation summaries between runs on the same sample?
# (Proxy: jaccard similarity on word sets)
# ---------------------------------------------------------------------------

def jaccard(a: str, b: str) -> float:
    wa, wb = set(a.lower().split()), set(b.lower().split())
    if not wa and not wb:
        return 1.0
    return len(wa & wb) / len(wa | wb)


# Build presentation lookup per (sample_id, run_label)
presentations: dict[tuple, str] = {}
for _, r in df.iterrows():
    key = (r["sample_id"], r["run_label"])
    for label, art in artifacts.items():
        if r["run_label"] == label:
            for result in art.get("results", []):
                if result.get("sample_id") == r["sample_id"]:
                    presentations[key] = result.get("insights", {}).get("clinical_presentation", "")

samples = df["sample_id"].unique()
consistency_rows = []
for sid in samples:
    c_text = presentations.get((sid, "candidate"), "")
    b_text = presentations.get((sid, "baseline"), "")
    if c_text and b_text:
        sim = jaccard(c_text, b_text)
        consistency_rows.append({"sample_id": sid, "jaccard_similarity": round(sim, 3),
                                  "candidate_words": len(c_text.split()),
                                  "baseline_words": len(b_text.split())})

if consistency_rows:
    cons_df = pd.DataFrame(consistency_rows)
    print("Cross-run clinical_presentation similarity (Jaccard on word sets)")
    print("Higher = more consistent insights across the two STT models.")
    display(cons_df)
    mean_sim = cons_df["jaccard_similarity"].mean()
    print(f"\nMean similarity: {mean_sim:.3f}")
    if mean_sim < 0.5:
        print("⚠  Low cross-run similarity suggests the insight model is sensitive to transcript differences.")
        print("   This means STT errors are propagating into clinical conclusions — a meaningful risk in production.")
    else:
        print("✓  High similarity suggests insights are robust to minor transcript differences between models.")
else:
    print("Not enough paired data to compute cross-run consistency. Run both candidate and baseline insights.")

In [ ]:
# ---------------------------------------------------------------------------
# Inference throughput
# ---------------------------------------------------------------------------
if not df.empty and df["elapsed_s"].notna().any():
    throughput = df[["run_label", "sample_id", "elapsed_s", "word_count"]].copy()
    throughput["words_per_sec"] = (throughput["word_count"] / throughput["elapsed_s"]).round(1)
    print("Inference throughput (transcript processing speed via Ollama)")
    display(throughput)
    print(f"\nMean elapsed: {throughput['elapsed_s'].mean():.1f}s per sample")
    print(f"Mean input rate: {throughput['words_per_sec'].mean():.0f} transcript-words/sec")